In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys, os
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from NNMFit.utilities import ScanHandler

# import plotting utils
sys.path.append( "/data/user/tvaneede/utils" )
from FlavourScansPlotting import get_contour_points, get_contour_points_angle

# import scanning points
from flavor_fracs import *
points = pd.read_pickle('/data/user/tvaneede/GlobalFit/custom_scan_flavor/default_custom_scan_points_flavor.pickle')
flavs = flavor_frac(np.asarray(points['astro_nue_ratio']),np.asarray(points['astro_nutau_ratio']))

# import plotting flavor triangle
sys.path.append( "/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/notebooks" )
from plot_utils_triangle import *


In [ ]:
plotting_path = "/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/notebooks/unblind/step2_hese_flavor_reunblind/flavor_triangle"
os.system(f"mkdir -p {plotting_path}")

In [ ]:
dag_path = "/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/dag_scans/flavor_globalfit/hese/unblind/step2_hese_flavor_reunblind/"
scan_dir_dict = {
    # previous scans without detector priors (`priors` key was not read by NNMFit):
    # "nbestfit50_fine_0" : f"{dag_path}/data_fits_flavor_triangle/nbestfit50_fine_0",
    # "nbestfit50_fine_1" : f"{dag_path}/data_fits_flavor_triangle/nbestfit50_fine_1",
    # "nbestfit50_fine_2" : f"{dag_path}/data_fits_flavor_triangle/nbestfit50_fine_2",
    # scan with the detector priors applied (fixed `prior` key)
    "nbestfit50_fixedprior_0" : f"{dag_path}/data_fits_flavor_triangle/nbestfit50_fixedprior_0",
}

In [ ]:
sys.path.insert(0, "/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/notebooks/unblind/step2_hese_flavor_reunblind")
from analysis_utils import load_triangle_scan

N_grid = 40 # 20 was standard
drop_unsuccessful = False
scan_dict = {}
data = {}
bestfits = {}
for name, scan_dir in scan_dir_dict.items():
    print(name)
    # all scan points (drop_unsuccessful = False): get_contour_points always drops fits flagged as failed,
    # which lie on the profile as well (see studies/asimov_closure.ipynb)
    scan_dict[name], data[name], bestfits[name] = load_triangle_scan(
        scan_dir, N_grid=N_grid, drop_unsuccessful=drop_unsuccessful, drop_unsuccessful_points=drop_unsuccessful)
    print(f"{data[name]['n_converged']} converged, {len(data[name]['ts'])} used scan points")

In [ ]:
# Combine scans: lowest TS per scan point, renormalized to the global best fit (see analysis_utils.combine_scans)
sys.path.insert(0, "/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/notebooks/unblind/step2_hese_flavor_reunblind")
from analysis_utils import combine_scans, load_triangle_scan

for name in scan_dir_dict:
    print(f"{name}: freefit llh = {bestfits[name]['llh']:.4f}")
data["combined"], bestfits["combined"] = combine_scans(list(scan_dir_dict), data, bestfits, N_grid)


In [ ]:
for name in data:

    f_e = bestfits[name]["astro_nue_ratio"] / (1 + bestfits[name]["astro_nue_ratio"] + bestfits[name]["astro_nutau_ratio"])
    f_mu = 1 / (1 + bestfits[name]["astro_nue_ratio"] + bestfits[name]["astro_nutau_ratio"])
    f_tau = bestfits[name]["astro_nutau_ratio"] / (1 + bestfits[name]["astro_nue_ratio"] + bestfits[name]["astro_nutau_ratio"])

    C = plot_contour( data = data,
                    name = name,
                    label = "Sensitivity",
                    levels = ["68%","90%","95%"],
                    title = "",
                    scenarios=True,
                    show_ts_labels=False,
                    bestfit=[f_tau,f_e],
                    savepath = f"{plotting_path}/flavor_triangle_{name}.png")


## Comparison with previous iteration


In [ ]:
previous_dag_path = "/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/dag_scans/flavor_globalfit/hese/unblind/step2_hese_flavor/"
previous_scan_dir_dict = {
    "previous_nbestfit50_fine_0" : f"{previous_dag_path}/data_fits_flavor_triangle/nbestfit50_fine_0",
    "previous_nbestfit50_fine_1" : f"{previous_dag_path}/data_fits_flavor_triangle/nbestfit50_fine_1",
    "previous_nbestfit50_fine_2" : f"{previous_dag_path}/data_fits_flavor_triangle/nbestfit50_fine_2",
}

for name, scan_dir in previous_scan_dir_dict.items():
    print(name)
    scan_dict[name], data[name], bestfits[name] = load_triangle_scan(
        scan_dir, N_grid=N_grid, drop_unsuccessful=drop_unsuccessful, drop_unsuccessful_points=drop_unsuccessful)

data["previous_combined"], bestfits["previous_combined"] = combine_scans(list(previous_scan_dir_dict), data, bestfits, N_grid)


In [ ]:
def bestfit_fractions(bf):
    norm = 1 + bf["astro_nue_ratio"] + bf["astro_nutau_ratio"]
    return bf["astro_nutau_ratio"] / norm, bf["astro_nue_ratio"] / norm  # f_tau, f_e

compare_names = ["previous_combined", "combined"]
compare_labels = ["Previous iteration", "This analysis"]
savepath = f"{plotting_path}/flavor_triangle_compare_previous.png"

C = compare_contours(data, names=compare_names, labels=compare_labels,
                     levels=["68%", "90%", "95%"], title="", scenarios=True)
ax = plt.gca()
for i, name in enumerate(compare_names):
    f_tau, f_e = bestfit_fractions(bestfits[name])
    print(f"{compare_labels[i]}: f_e = {f_e:.3f}, f_mu = {1 - f_e - f_tau:.3f}, f_tau = {f_tau:.3f}")
    ax.scatter([f_tau], [f_e], marker='X', facecolor=colours[i], edgecolor='k', lw=0.5, s=80, zorder=10)
plt.savefig(savepath, bbox_inches='tight')
plt.show()
